In [ ]:
import os
import subprocess
from pathlib import Path

import anndata as ad

from batch_integration.multisample_utils import (
    preprocess_genes,
    preprocess_neighbors,
    preprocess_scaling,
)
from utils import DATA_DIR, RESULT_DIR, SHELL, activate_env_cmd

In [ ]:
# Settings
seed = 42

SINGLE_DIR = "single_sample"
PATIENT_DIR = "patient"
ALL_DIR = "all"

In [ ]:
libd_dlpfc_dir = DATA_DIR / "LIBD_DLPFC"
libd_dlpfc_results = RESULT_DIR / "LIBD_DLPFC"

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

In [ ]:
adata = ad.read_h5ad(libd_dlpfc_dir / "raw.h5ad")

del adata.obsm
del adata.uns
del adata.layers

adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy().astype(int)

adata.var = adata.var[["gene_name", "type", "gene_biotype"]]

adata.obs = adata.obs[["sample_id", "subject", "discard", "spatialLIBD"]].rename(
    columns={"spatialLIBD": "label"}
)

adata.X = adata.X.tocsr()

# Preprocess

## Single sample

In [ ]:
out_dir = libd_dlpfc_dir / SINGLE_DIR
out_dir.mkdir(exist_ok=True, parents=True)

for patient_sample, idx in adata.obs.groupby(
    ["subject", "sample_id"], observed=True
).indices.items():
    sample = "_".join(patient_sample)
    adata_sample = adata[idx, :].copy()
    adata_sample.uns["n_clusters"] = adata_sample.obs["label"].nunique()

    preprocess_genes(adata_sample, batch_key=None)
    preprocess_scaling(adata_sample, seed=seed)

    adata_sample.write(out_dir / f"{sample}.h5ad")

## Patient-wise

In [ ]:
out_dir = libd_dlpfc_dir / PATIENT_DIR
out_dir.mkdir(exist_ok=True, parents=True)

for patient, idx in adata.obs.groupby("subject", observed=True).indices.items():
    adata_patient = adata[idx, :].copy()
    adata_patient.uns["n_clusters"] = adata_patient.obs["label"].nunique()
    adata_patient.uns["batch_keys"] = ["sample_id"]
    adata_patient.obs_names_make_unique()

    preprocess_genes(adata_patient, batch_key=None)
    preprocess_scaling(adata_patient, seed=seed)

    adata_patient.write(out_dir / f"{patient}.h5ad")

## All

In [ ]:
out_dir = libd_dlpfc_dir / ALL_DIR
out_dir.mkdir(exist_ok=True, parents=True)

adata.obs_names_make_unique()
adata.uns["n_clusters"] = adata.obs["label"].nunique()
adata.uns["batch_keys"] = ["sample_id", "subject"]

preprocess_genes(adata, batch_key=adata.uns["batch_keys"][0])
preprocess_scaling(adata, seed=seed)

adata.write(out_dir / "LIBD_DLPFC.h5ad")

# Batch integration

In [ ]:
script_dir = Path("batch_integration")

## Single sample (i.e. no integration)

In [ ]:
conda_cmd = activate_env_cmd("no_batch_integration")

In [ ]:
file_dir = libd_dlpfc_dir / SINGLE_DIR
script_path = script_dir / "no_batch_integration.py"

for h5ad in file_dir.glob("*.h5ad"):
    name = h5ad.stem
    out_dir = libd_dlpfc_results / h5ad.parent.name
    out_dir.mkdir(exist_ok=True, parents=True)
    out_file = out_dir / f"{name}.npz"

    cmd = f"{script_path} {h5ad} {out_file}"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

## Patient-wise / All

In [ ]:
methods = [
    "no_batch_integration",
    "bbknn",
    "harmony",
    "scvi",
]

In [ ]:
files = list((libd_dlpfc_dir / PATIENT_DIR).glob("*.h5ad")) + list(
    (libd_dlpfc_dir / ALL_DIR).glob("*.h5ad")
)

In [ ]:
for h5ad in files:
    name = h5ad.stem

    for method in methods:
        out_dir = libd_dlpfc_results / method / h5ad.parent.name
        out_dir.mkdir(exist_ok=True, parents=True)
        out_file = out_dir / f"{name}.npz"

        script_path = script_dir / f"{method}.py"

        conda_cmd = activate_env_cmd(method)
        cmd = f"{script_path} {h5ad} {out_file}"

        # because bbknn uses n neighbors per batch we decrease the number of neighbors
        # so its roughly equal to the rest
        if method == "bbknn":
            n = 5  # change based on within or across patients?
            cmd += f" --n_neighs {n}"

        subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Clustering

In [ ]:
script_dir = Path("clustering")

In [ ]:
conda_cmd = activate_env_cmd("spatialleiden_multisample")
script_path = script_dir / "spatialleiden_multisample.py"

In [ ]:
layer_ratio = 0.8

## Single sample

In [ ]:
file_dir = libd_dlpfc_dir / SINGLE_DIR

for h5ad in file_dir.glob("*.h5ad"):
    name = h5ad.stem
    out_dir = libd_dlpfc_results / h5ad.parent.name
    neighbor_file = out_dir / f"{name}.npz"
    out_file = out_dir / f"{name}.parquet"

    cmd = f"{script_path} {h5ad} {neighbor_file} {out_file}"
    # cmd+=f" --n_cluster {n_cluster}"
    cmd += f" --weight {layer_ratio}"
    cmd += " --coord_type grid"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

## Patient-wise / All

In [ ]:
for file in libd_dlpfc_results.glob("*/*/*.npz"):
    name = file.stem
    batch = file.parent.name
    method = file.parents[1].name

    h5ad = libd_dlpfc_dir / batch / f"{name}.h5ad"
    out_file = file.parent / f"{name}.parquet"

    cmd = f"{script_path} {h5ad} {file} {out_file}"
    # bbknn has different number of neighbors therefore different weight is required
    if method == "bbknn":
        cmd += f" --weight {layer_ratio}"
    else:
        cmd += " --weight 1.2"
    cmd += " --coord_type grid"
    cmd += " --use_batch"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Evaluation

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.metrics import adjusted_rand_score

In [ ]:
aris = list()

for file in libd_dlpfc_results.glob("*/*/*.parquet"):
    tool = file.parents[1].name
    batch = file.parent.name
    name = file.stem
    labels = pd.read_parquet(file).astype({"spatialleiden": "category"})
    grouping = [x for x in ["subject", "sample_id"] if x in labels.columns]

    for g, df in labels.groupby(grouping, observed=True):
        sample = g[1] if len(g) > 1 else g[0]
        patient = g[0] if len(g) > 1 else name
        ari = adjusted_rand_score(df["label"].cat.codes, df["spatialleiden"].cat.codes)
        aris.append(
            {
                "ARI": ari,
                "sample": sample,
                "patient": patient,
                "batch": batch,
                "tool": tool,
            }
        )

for file in libd_dlpfc_results.glob("*/*.parquet"):
    patient, sample = file.stem.split("_")
    labels = pd.read_parquet(file, columns=["label", "spatialleiden"]).astype(
        "category"
    )
    ari = adjusted_rand_score(
        labels["label"].cat.codes, labels["spatialleiden"].cat.codes
    )
    aris.append(
        {
            "ARI": ari,
            "sample": sample,
            "batch": "na",
            "tool": "no_batch_integration",
            "patient": patient,
        }
    )

aris = pd.DataFrame(aris)

In [ ]:
aris = aris.assign(
    tool=lambda df: df["tool"].replace(
        {
            "bbknn": "BBKNN",
            "harmony": "Harmony",
            "scvi": "scVI",
            "no_batch_integration": "no integration",
        }
    )
)

In [ ]:
g = sns.catplot(
    data=aris,
    kind="swarm",
    sharex=False,
    x="tool",
    y="ARI",
    hue="patient",
    col="batch",
    aspect=0.8,
    dodge=True,
)
_ = g.set(ylim=(0, 0.8))